# Leakage-safe pipelines and baselines

**P0 Essential · D3 Synthesis · 110 minutes**

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from pathlib import Path

def locate(relative: str, local_name: str | None = None) -> Path:
    candidates = []
    if local_name:
        candidates.append(Path.cwd() / local_name)
    candidates.extend(root / relative for root in [Path.cwd(), *Path.cwd().parents])
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"Cannot find {relative}. Run from the course clone or place the downloaded data beside this notebook."
    )


In [ ]:
path = locate('datasets/teaching/predictive-maintenance/observations.csv', 'observations.csv')
machines = pd.read_csv(path)
machines['Machine failure'].value_counts(normalize=True).sort_index()

## Feature-time audit

`TWF`, `HDF`, `PWF`, `OSF`, and `RNF` are failure-mode indicators used to construct the aggregate target. They are not predictors available before the outcome. `UDI` and `Product ID` are identifiers, not default predictive features.

In [ ]:
def make_split(frame: pd.DataFrame):
    target = 'Machine failure'
    excluded = ['UDI','Product ID',target,'TWF','HDF','PWF','OSF','RNF']
    X = frame.drop(columns=excluded)
    y = frame[target]
    return train_test_split(X, y, test_size=0.25, random_state=20260718, stratify=y)

def make_pipeline() -> Pipeline:
    numeric = ['Air temperature [K]','Process temperature [K]','Rotational speed [rpm]',
               'Torque [Nm]','Tool wear [min]']
    categorical = ['Type']
    preprocessing = ColumnTransformer([
        ('num', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), numeric),
        ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')),
                          ('encode', OneHotEncoder(handle_unknown='ignore'))]), categorical),
    ])
    return Pipeline([('preprocess', preprocessing),
                     ('model', LogisticRegression(max_iter=1000, class_weight='balanced'))])

In [ ]:
X_train, X_test, y_train, y_test = make_split(machines)
assert not {'TWF','HDF','PWF','OSF','RNF','Machine failure'} & set(X_train.columns)
model = make_pipeline().fit(X_train, y_train)
dummy = DummyClassifier(strategy='prior').fit(X_train, y_train)
model_ap = average_precision_score(y_test, model.predict_proba(X_test)[:, 1])
dummy_ap = average_precision_score(y_test, dummy.predict_proba(X_test)[:, 1])
assert model_ap > dummy_ap
{'model_pr_auc': model_ap, 'dummy_pr_auc': dummy_ap}

## Transfer

Repair a version that imputes globally and retains `HDF`. Explain the split and feature-time boundary, add a leakage assertion, and state why this synthetic random split is not evidence of future factory performance.